In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Akash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Akash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Akash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## loading dataset

In [2]:
df = pd.read_csv('C:/Users/Akash/Downloads/IMDB_Dataset.csv')

## Data exploration

In [4]:
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
display(df.head())

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Positive Review:")
print(df[df['sentiment'] == 'positive']['review'].iloc[0][:500], "...\n")
print("Sample Negative Review:")
print(df[df['sentiment'] == 'negative']['review'].iloc[0][:500], "...")

Dataset Shape: (50000, 2)

First 5 rows of the dataset:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Positive Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ ...

Sample Negative Review:
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a dra

## Preprocessing Function

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercasing 
    text = text.lower()
    
    #  Removing URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Removing HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Removing special characters and punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenization 
    tokens = word_tokenize(text)
    
    # Removing stopwords 
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # Join tokens 
    return " ".join(cleaned_tokens)

## Apply Preprocessing

In [6]:
print("Starting preprocessing...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete!")

display(df[['review', 'cleaned_review', 'sentiment']].head())

Starting preprocessing...
Preprocessing complete!


,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode y...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,positive


## Train-Test Split

In [7]:
df['sentiment_encoded'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'], 
    df['sentiment_encoded'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['sentiment_encoded']
)

## Feature Engineering - Bag of Words

In [8]:
print("Applying Bag of Words...")
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

Applying Bag of Words...


## Feature Engineering - TF-IDF

In [17]:
print("Applying TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

Applying TF-IDF...


## Evaluation Function

In [10]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test, model_name, feature_type):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"--- {model_name} with {feature_type} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}\n")
    
    return {'Model': model_name, 'Features': feature_type, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

## Initialize Models

In [12]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []

In [13]:
print("Evaluating Models using Bag of Words (BoW):\n")
for name, model in models.items():
    res = train_and_evaluate(model, X_train_bow, X_test_bow, y_train, y_test, name, "BoW")
    results.append(res)

Evaluating Models using Bag of Words (BoW):

--- Logistic Regression with BoW ---
Accuracy : 0.8730
Precision: 0.8686
Recall   : 0.8790
F1 Score : 0.8738

--- Naive Bayes with BoW ---
Accuracy : 0.8448
Precision: 0.8462
Recall   : 0.8428
F1 Score : 0.8445

--- Decision Tree with BoW ---
Accuracy : 0.7254
Precision: 0.7279
Recall   : 0.7200
F1 Score : 0.7239



In [18]:
print("Evaluating Models using TF-IDF:\n")
for name, model in models.items():
    res = train_and_evaluate(model, X_train_tfidf, X_test_tfidf, y_train, y_test, name, "TF-IDF")
    results.append(res)

Evaluating Models using TF-IDF:

--- Logistic Regression with TF-IDF ---
Accuracy : 0.8876
Precision: 0.8791
Recall   : 0.8988
F1 Score : 0.8888

--- Naive Bayes with TF-IDF ---
Accuracy : 0.8544
Precision: 0.8476
Recall   : 0.8642
F1 Score : 0.8558

--- Decision Tree with TF-IDF ---
Accuracy : 0.7262
Precision: 0.7278
Recall   : 0.7226
F1 Score : 0.7252



## Results Comparison

In [19]:
results_df = pd.DataFrame(results)
display(results_df.sort_values(by="F1", ascending=False))

,Model,Features,Accuracy,Precision,Recall,F1
3,Logistic Regression,TF-IDF,0.8876,0.879108,0.8988,0.888845
6,Logistic Regression,TF-IDF,0.8876,0.879108,0.8988,0.888845
9,Logistic Regression,TF-IDF,0.8876,0.879108,0.8988,0.888845
0,Logistic Regression,BoW,0.8730,0.868577,0.8790,0.873757
4,Naive Bayes,TF-IDF,0.8544,0.847587,0.8642,0.855813
7,Naive Bayes,TF-IDF,0.8544,0.847587,0.8642,0.855813
10,Naive Bayes,TF-IDF,0.8544,0.847587,0.8642,0.855813
1,Naive Bayes,BoW,0.8448,0.846185,0.8428,0.844489
5,Decision Tree,TF-IDF,0.7262,0.727840,0.7226,0.725211
8,Decision Tree,TF-IDF,0.7262,0.727840,0.7226,0.725211
